<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "Hugging Face is creating great tools for NLP!"
print(f"Phrase choisie : {sample_sentence}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Phrase choisie : Hugging Face is creating great tools for NLP!


In [2]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=20,
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    # Highlight special tokens
    display_token = f"***{token}***" if token in tokenizer.all_special_tokens else token
    print(f"{idx:>5} | {display_token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

index | token        | id
-------------------------
    0 | ***[CLS]***  |   101
    1 | hugging      | 17662
    2 | face         |  2227
    3 | is           |  2003
    4 | creating     |  4526
    5 | great        |  2307
    6 | tools        |  5906
    7 | for          |  2005
    8 | nl           | 17953
    9 | ##p          |  2361
   10 | !            |   999
   11 | ***[SEP]***  |   102
   12 | ***[PAD]***  |     0
   13 | ***[PAD]***  |     0
   14 | ***[PAD]***  |     0
   15 | ***[PAD]***  |     0
   16 | ***[PAD]***  |     0
   17 | ***[PAD]***  |     0
   18 | ***[PAD]***  |     0
   19 | ***[PAD]***  |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (11, '[SEP]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]')]


### Exercise 1 reflection
- **[CLS] & [SEP]**: Le jeton `[CLS]` (Classification) est inséré au début de la séquence pour représenter l'agrégation de toute la phrase pour les tâches de classification. Le jeton `[SEP]` (Separator) marque la fin d'une phrase ou la séparation entre deux phrases.
- **Attention Mask**: Le masque d'attention indique au modèle quels jetons doivent être ignorés (valeur 0 pour le `[PAD]`) afin que le mécanisme d'auto-attention ne soit pas influencé par le remplissage inutile.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [3]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "I am absolutely amazed by how powerful and easy to use Transformers are!"
prediction = sentiment_pipeline(sentence)
print(f"Phrase : {sentence}")
print(f"Résultat : {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Phrase : I am absolutely amazed by how powerful and easy to use Transformers are!
Résultat : [{'label': 'POSITIVE', 'score': 0.9996205568313599}]


### Exercise 2 reflection
- **Attente**: Le label devrait être `POSITIVE` car la phrase utilise des adjectifs forts comme 'amazed' et 'powerful'.
- **Confiance**: Le score proche de 0.99+ indique que le modèle est extrêmement confiant. Ce score représente la probabilité assignée à la classe prédite après passage dans la fonction softmax.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.max_length = max_length
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        return self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)

        score, index = torch.max(probs, dim=-1)
        label = self.model.config.id2label[index.item()]

        return {"label": label, "probability": score.item()}

In [5]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "This tutorial is incredibly helpful and clear!",
    "I am very disappointed with the quality of this product."
]
for text in samples:
    result = analyzer.predict(text)
    print(f"Texte: {text}")
    print(f"Prédiction: {result}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Texte: This tutorial is incredibly helpful and clear!
Prédiction: {'label': 'POSITIVE', 'probability': 0.9997262358665466}

Texte: I am very disappointed with the quality of this product.
Prédiction: {'label': 'NEGATIVE', 'probability': 0.999798595905304}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [8]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def recognize(self, text: str):
        # return_offsets_mapping permet de retrouver la position exacte dans le texte original
        inputs = self.tokenizer(text, return_tensors="pt", return_offsets_mapping=True).to(self.device)
        offset_mapping = inputs.pop("offset_mapping")[0].tolist()

        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=2)[0].tolist()

        entities = []
        current_entity = None

        for i, pred_idx in enumerate(predictions):
            label = self.model.config.id2label[pred_idx]
            start, end = offset_mapping[i]

            # Ignorer les jetons spéciaux (ID 0 dans offset_mapping)
            if start == end == 0: continue

            if label.startswith("B-"):
                if current_entity: entities.append(current_entity)
                current_entity = {"text": text[start:end], "label": label[2:], "start": start, "end": end}
            elif label.startswith("I-") and current_entity and label[2:] == current_entity["label"]:
                # Fusionner avec l'entité en cours
                current_entity["text"] = text[current_entity["start"]:end]
                current_entity["end"] = end
            else:
                if current_entity: entities.append(current_entity)
                current_entity = None

        if current_entity: entities.append(current_entity)
        return entities

In [9]:
ner = BERTNamedEntityRecognizer()
sample_text = "Hugging Face is a company based in New York City and Paris."
results = ner.recognize(sample_text)

print(f"Texte d'origine : {sample_text}\n")
print(f"{'ENTITÉ FUSIONNÉE':<20} | {'TYPE':<10} | {'INDICES'}")
print("-" * 50)
for ent in results:
    print(f"{ent['text']:<20} | {ent['label']:<10} | {ent['start']}:{ent['end']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texte d'origine : Hugging Face is a company based in New York City and Paris.

ENTITÉ FUSIONNÉE     | TYPE       | INDICES
--------------------------------------------------
Hugging Face         | ORG        | 0:12
New York City        | LOC        | 35:48
Paris                | LOC        | 53:58


### Visualisation du découpage brut (Intermédiaire)
Voici comment BERT segmente le texte en sous-unités (WordPieces) avant que nous n'appliquions notre logique de fusion :

In [10]:
inputs_raw = ner.tokenizer(sample_text, return_tensors="pt").to(ner.device)
tokens_raw = ner.tokenizer.convert_ids_to_tokens(inputs_raw["input_ids"][0])

with torch.no_grad():
    outputs_raw = ner.model(**inputs_raw)
    predictions_raw = torch.argmax(outputs_raw.logits, dim=2)[0].tolist()

print(f"Phrase : {sample_text}\n")
print(f"{'JETON (TOKEN)':<15} | {'LABEL BIO'}")
print("-" * 30)
for token, pred_idx in zip(tokens_raw, predictions_raw):
    label = ner.model.config.id2label[pred_idx]
    # On ignore les jetons spéciaux pour la clarté
    if token in ["[CLS]", "[SEP]"]: continue
    print(f"{token:<15} | {label}")

Phrase : Hugging Face is a company based in New York City and Paris.

JETON (TOKEN)   | LABEL BIO
------------------------------
Hu              | B-ORG
##gging         | I-ORG
Face            | I-ORG
is              | O
a               | O
company         | O
based           | O
in              | O
New             | B-LOC
York            | I-LOC
City            | I-LOC
and             | O
Paris           | B-LOC
.               | O


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encodeur uniquement (Auto-encoding) | Décodeur uniquement (Auto-regressive) |
| Primary purpose | Compréhension du langage et contexte bidirectionnel | Génération de texte et prédiction du mot suivant |
| Typical use cases | Classification, NER, Questions-Réponses | Chatbots, Création de contenu, Résumé |
| Strengths | Contexte global (gauche et droite) simultané | Capacité de génération fluide et cohérente |
| Weaknesses | Ne peut pas générer de texte de manière efficace | Contexte unidirectionnel (ne regarde que le passé) |

## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **Encodage**: BERT transforme les documents et la requête utilisateur en vecteurs denses (embeddings) capturant le sens sémantique plutôt que de simples mots-clés.
2. **Stockage et Recherche**: Ces vecteurs sont stockés dans une base de données vectorielle. Lors d'une requête, on utilise la similarité cosinus pour retrouver les passages dont le vecteur BERT est le plus proche de celui de la question.
3. **Passage au Générateur**: Les segments récupérés sont concaténés à la question initiale pour former un 'super-prompt'. Ce contexte enrichi est alors envoyé à un modèle comme GPT pour générer une réponse sourcée.
4. **Application**: Un système de support technique pour une entreprise complexe (ex: aéronautique) où BERT retrouve les bonnes pages de manuels techniques volumineux pour que GPT rédige une procédure d'assistance précise.